In [3]:
pip install sympy numpy scipy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 201.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 72.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 210.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 143.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 179.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 269.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 184.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 155.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [matplotlib]1 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import Modules
%reset -f 
import sympy as sy 
from IPython.display import display, Math
from sympy.utilities.codegen import codegen
import numpy as np
from scipy.integrate import solve_ivp

In [4]:
# --- Define Symbols --- #
# Time
t = sy.symbols('t')

# Define State Space [xc, dxc, theta, dtheta] ([cart position, cart velocity, pendulum angle, pendulum angle velocity])
xc     = sy.Function('xc')(t) 
theta  = sy.Function('theta')(t) 

# Their derivatives 
dxc     = sy.diff(xc, t) 
dtheta  = sy.diff(theta, t) 

# System
mc, mp, lpC, g = sy.symbols('m_c m_p l_s g', real=True)
Izz = sy.symbols('I_zz', real=True)

# Forces
F = sy.symbols('F', real=True) 
kc, kp = sy.symbols('kc, kp', real=True)

In [5]:
# --- Derive Kinematics --- #

# Rotationsmatrix

#R01 = sy.Matrix([[sy.cos(sy.pi/2-theta),-sy.sin(sy.pi/2-theta),0],[sy.sin(sy.pi/2-theta),sy.cos(sy.pi/2-theta),0],[0,0,1]])
R01 = sy.Matrix([[sy.cos(theta),-sy.sin(theta),0],[sy.sin(theta),sy.cos(theta),0],[0,0,1]])

# Position Components of the Cart and Pendulum 
pCart     = sy.Matrix([xc, 0, 0])
pPendulum = pCart+R01*sy.Matrix([0,lpC, 0])

vCart     = pCart.diff(t)
vPendulum = pPendulum.diff(t)

S01 = sy.simplify(R01.diff(t)*R01.transpose())
om01 = sy.Matrix([S01[2,1],S01[0,2],S01[1,0]])

display(Math(f"pC = {sy.latex(pCart)}")) 
display(Math(f"pP = {sy.latex(pPendulum)}"))

display(Math(f"vC = {sy.latex(vCart)}")) 
display(Math(f"vP = {sy.latex(vPendulum)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
# --- Energies --- #

# Kinetic
Tc = sy.Rational(1,2) * mc * vCart[0] * vCart[0]

Ttrans = sy.Rational(1,2) * mp *  (vPendulum[0]**2 + vPendulum[1]**2)
Trot   = sy.Rational(1,2) * Izz * dtheta**2

Tp = Ttrans + Trot

# Potential
Vc = mc * g * pCart[1]
Vp = mp * g * pPendulum[1]

# Total
T = sy.simplify(Tc + Tp)
V = sy.simplify(Vc + Vp)

display(Math(f"T = {sy.latex(Trot)}")) 
display(Math(f"V = {sy.latex(V)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [7]:
# --- Raleigh Dissipation --- #

R = sy.Rational(1,2) * kc * vCart[0]**2 + sy.Rational(1,2) * kp * dtheta**2

dR_vc = sy.diff(R, dxc)
dR_vp = sy.diff(R, dtheta)

display(Math(f"dRvC = {sy.latex(dR_vc)}")) 
display(Math(f"dRcP = {sy.latex(dR_vp)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
# --- Langrange Equation --- #

# Generalized coordinates
q = [xc, theta]
dq = [dxc, dtheta]

# Langrangian Equation
L = sy.simplify(T-V)

# Subsystem 1 - Cart
dL_xc   = sy.diff(L, q[0])
dL_vc   = sy.diff(L, dq[0])
dtdL_vc = sy.diff(dL_vc, t)

# Subsystem 2 - Pendulum
dL_xp   = sy.diff(L, q[1])
dL_vp   = sy.diff(L, dq[1])
dtdL_vp = sy.diff(dL_vp, t)

Lc =  sy.simplify(dtdL_vc - dL_xc - F + dR_vc)
Lp =  sy.simplify(dtdL_vp - dL_xp + dR_vp)

display(Math(f"0 = {sy.latex(Lc)}")) 
display(Math(f"0 = {sy.latex(Lp)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [9]:
## Reshape Equations 
# Substitute Functions with Placeholder Vars 
xcS, thetaS = sy.symbols('x_c thetaS') 
dxc_S, dtheta_S = sy.symbols('dx_c dtheta') 
ddxc, ddtheta = sy.symbols('ddx_c ddtheta') 

LI = Lc.subs(
    {xc: xcS, 
     theta: thetaS, 
     sy.diff(xc, (t,1)): dxc_S, 
     sy.diff(theta, (t,1)): dtheta_S, 
     sy.diff(xc, (t,2)): ddxc, 
     sy.diff(theta, (t,2)): ddtheta})

LII = Lp.subs(
    {xc: xcS, 
     theta: thetaS, 
     sy.diff(xc, (t,1)): dxc_S, 
     sy.diff(theta, (t,1)): dtheta_S, 
     sy.diff(xc, (t,2)): ddxc, 
     sy.diff(theta, (t,2)): ddtheta}) 

# Isolate Accelerations 
solution = sy.solve([LI, LII], [ddxc, ddtheta], simplify=True) 

ddxc_sol = solution[ddxc] 
ddtheta_sol = solution[ddtheta] 

# Convert to LaTeX string 
display(Math(f"ddxc = {sy.latex(ddxc_sol)}"))
display(Math(f"ddtheta = {sy.latex(ddtheta_sol)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [33]:
import re

# Generate Python code for expressions
eq1_code = sy.pycode(ddxc_sol)
eq2_code = sy.pycode(ddtheta_sol)

eq1_code = sy.pycode(ddxc_sol).replace("math.", "np.")
eq2_code = sy.pycode(ddtheta_sol).replace("math.", "np.")


def remove_eq_functions(file_path="model/pendulum.py"):
    # Read the file content
    with open(file_path, "r") as f:
        content = f.read()

    # Pattern to match a full function block (eq1 or eq2)
    def remove_function_block(content, func_name):
        pattern = rf"def {func_name}\(.*?\n(?:(?!^def ).*\n)*"
        return re.sub(pattern, "", content, flags=re.MULTILINE)

    # Remove eq1 and eq2 if they exist
    content = remove_function_block(content, "eq1")
    content = remove_function_block(content, "eq2")

    # Write back the cleaned content
    with open(file_path, "w") as f:
        f.write(content)

remove_eq_functions("model/pendulum.py")

with open("model/pendulum.py", "a") as f:
    f.write("\n") 
    f.write("def eq1(xc, dx_c, thetaS, dtheta, m_c, m_p, l_s, g, I_zz, kc, kp, F):\n")
    f.write(f"    return {eq1_code}\n\n")
    
with open ("model/pendulum.py", "a") as f:
    f.write("def eq2(xc, dx_c, thetaS, dtheta, m_c, m_p, l_s, g, I_zz, kc, kp, F):\n")
    f.write(f"    return {eq2_code}\n")